# GFF Parsing with Biopython (BCBio)

This notebook demonstrates how to parse a GFF3 file using `bcbio-gff` as suggested in the [Biopython Wiki](https://biopython.org/wiki/GFF_Parsing).

In [ ]:
from BCBio import GFF
from Bio import SeqIO
import os
import pandas as pd

# Set the path to the GFF file
gff_file = "../viral_data_all/ncbi_dataset/data/GCF_000864765.1/genomic.gff"

if not os.path.exists(gff_file):
    print(f"Error: {gff_file} not found.")
else:
    print(f"Found GFF file: {gff_file}")

Found GFF file: ../viral_complete_subset_data/ncbi_dataset/data/GCF_000864765.1/genomic.gff


## Recursive Search for "slip"

GFF files often have nested features (e.g., CDS inside an mRNA inside a gene). To find all instances of "slip", we need to search recursively through all levels of features.

In [ ]:
def search_features_recursive(features, search_term, record_id):
    results = []
    for feature in features:
        match = False
        
        # Check feature type
        if search_term in str(feature.type).lower():
            match = True
        
        # Check all qualifiers (attributes like 'Note', 'exception', etc.)
        if not match:
            for key, values in feature.qualifiers.items():
                for val in values:
                    if search_term in str(val).lower():
                        match = True
                        break
                if match: break
        
        if match:
            results.append({
                "record_id": record_id,
                "type": feature.type,
                "start": int(feature.location.start),
                "end": int(feature.location.end),
                "strand": feature.location.strand,
                "qualifiers": feature.qualifiers
            })
        
        # Recursively search through sub-features (children)
        if hasattr(feature, 'sub_features') and feature.sub_features:
            results.extend(search_features_recursive(feature.sub_features, search_term, record_id))
            
    return results

Found 2 total features matching 'slip'.


,record_id,type,start,end,strand
0,NC_001802.1,CDS,335,1637,1
1,NC_001802.1,CDS,1636,4642,1



Full attributes of the first match:
{'Dbxref': ['GenBank:NP_057849.4', 'GeneID:155348'],
 'ID': ['cds-NP_057849.4'],
 'Name': ['NP_057849.4'],
 'Note': ['Pr160'],
 'Parent': ['gene-HIV1gp1'],
 'exception': ['ribosomal slippage'],
 'gbkey': ['CDS'],
 'gene': ['gag-pol'],
 'locus_tag': ['HIV1gp1'],
 'phase': ['0'],
 'product': ['Gag-Pol'],
 'protein_id': ['NP_057849.4'],
 'source': ['RefSeq']}


In [22]:
all_slip_matches = []
search_term = "slip"

with open(gff_file) as in_handle:
    for record in GFF.parse(in_handle):
        matches = search_features_recursive(record.features, search_term, record.id)
        all_slip_matches.extend(matches)

print(f"Found {len(all_slip_matches)} total features matching '{search_term}'.")

if all_slip_matches:
    df_slip = pd.DataFrame(all_slip_matches)
    display(df_slip.drop(columns=['qualifiers']).head())
    
    print("\nFull attributes of the first match:")
    import pprint
    pprint.pprint(all_slip_matches[0]['qualifiers'])

Found 2 total features matching 'slip'.


,record_id,type,start,end,strand
0,NC_001802.1,CDS,335,1637,1
1,NC_001802.1,CDS,1636,4642,1



Full attributes of the first match:
{'Dbxref': ['GenBank:NP_057849.4', 'GeneID:155348'],
 'ID': ['cds-NP_057849.4'],
 'Name': ['NP_057849.4'],
 'Note': ['Pr160'],
 'Parent': ['gene-HIV1gp1'],
 'exception': ['ribosomal slippage'],
 'gbkey': ['CDS'],
 'gene': ['gag-pol'],
 'locus_tag': ['HIV1gp1'],
 'phase': ['0'],
 'product': ['Gag-Pol'],
 'protein_id': ['NP_057849.4'],
 'source': ['RefSeq']}
